# NeuralProphet for Multivariate Time Series

We quickly test how effective Neural Prophet is at forecasting multivariate time series. 

## Packages

In [23]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



## Data Preparation

In [24]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']

df = rs.copy()

## Model

In [79]:
import pandas as pd
import matplotlib.pyplot as plt
from neuralprophet import NeuralProphet

# Boroughs
BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']

# Ensure datetime
df['ds'] = pd.to_datetime(df['ds'])

# Create full date range
full_dates = pd.date_range('2020-01-01', '2026-02-28', freq='D')
full_index = pd.MultiIndex.from_product([full_dates, BOROUGHS], names=['ds','borough'])

# Reindex and fill missing
df_full = df.set_index(['ds','borough']).reindex(full_index).reset_index()
df_full['y'] = df_full['y'].fillna(0)
df_full = df_full.rename(columns={'borough':'ID'})

# Train NeuralProphet
m = NeuralProphet(
    yearly_seasonality=True,
    weekly_seasonality=True
)
m.fit(df_full, freq='D')

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined 

Finding best initial lr:   0%|          | 0/251 [00:00<?, ?it/s]

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/utilities/cloud_io.py:51: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to thi

Training: 0it [00:00, ?it/s]

,MAE,RMSE,Loss,RegLoss,epoch
0,0.658977,0.809130,0.391009,0.0,0
1,0.306135,0.388491,0.140338,0.0,1
2,0.183102,0.241861,0.062440,0.0,2
3,0.179495,0.237630,0.060309,0.0,3
4,0.178440,0.237078,0.059890,0.0,4
5,0.179375,0.238401,0.060445,0.0,5
6,0.179313,0.239224,0.060570,0.0,6
7,0.181161,0.240636,0.061570,0.0,7
8,0.179626,0.238748,0.060667,0.0,8
9,0.181301,0.240076,0.061341,0.0,9


In [84]:
# Forecast horizon
forecast_horizon = 14

# Dictionary to store 14-day forecasts per borough
forecast_by_borough = {}

for b in BOROUGHS:
    # Filter data for this borough
    df_b = df_full[df_full['ID'] == b].copy()
    
    # Make future dataframe for 14 days
    future_b = m.make_future_dataframe(df_b, periods=forecast_horizon, n_historic_predictions=0)
    
    # Predict using the existing model
    forecast_b = m.predict(future_b)
    
    # Extract only the 14-day forecast (future beyond training)
    forecast_14d_b = forecast_b[forecast_b['ds'] > df_b['ds'].max()][['ds', 'yhat1']].reset_index(drop=True)
    
    # Store in dictionary
    forecast_by_borough[b] = forecast_14d_b

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 88it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 88it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 88it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 88it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 92.857% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 88it [00:00, ?it/s]

In [85]:
forecast_by_borough

{'BRONX':            ds     yhat1
 0  2026-03-01  2.573042
 1  2026-03-02  6.515988
 2  2026-03-03  6.409023
 3  2026-03-04  6.217506
 4  2026-03-05  5.967527
 5  2026-03-06  5.099592
 6  2026-03-07  2.611990
 7  2026-03-08  2.821991
 8  2026-03-09  6.757502
 9  2026-03-10  6.642254
 10 2026-03-11  6.441811
 11 2026-03-12  6.182206
 12 2026-03-13  5.302453
 13 2026-03-14  2.800583,
 'BROOKLYN':            ds      yhat1
 0  2026-03-01   6.094047
 1  2026-03-02  15.432605
 2  2026-03-03  15.179264
 3  2026-03-04  14.725672
 4  2026-03-05  14.133616
 5  2026-03-06  12.077982
 6  2026-03-07   6.186292
 7  2026-03-08   6.683662
 8  2026-03-09  16.004610
 9  2026-03-10  15.731655
 10 2026-03-11  15.256920
 11 2026-03-12  14.642066
 12 2026-03-13  12.558442
 13 2026-03-14   6.632959,
 'MANHATTAN':            ds      yhat1
 0  2026-03-01   4.198121
 1  2026-03-02  10.631350
 2  2026-03-03  10.456827
 3  2026-03-04  10.144352
 4  2026-03-05   9.736491
 5  2026-03-06   8.320388
 6  2026-03-07   

In [ ]:
# Create a copy to work with
data = forecast_by_borough.copy()

for borough in data:
    df = data[borough]
    for i in range(len(df)):
        df.loc[i, 'yhat1'] = round(df.loc[i, 'yhat1'])


wide_df = pd.DataFrame({'ds': data['BRONX']['ds']})
for borough in data:
    wide_df[borough] = data[borough]['yhat1']

wide_df

,ds,BRONX,BROOKLYN,MANHATTAN,QUEENS,STATEN ISLAND
0,2026-03-01,3.0,6.0,4.0,3.0,1.0
1,2026-03-02,7.0,15.0,11.0,7.0,2.0
2,2026-03-03,6.0,15.0,10.0,7.0,2.0
3,2026-03-04,6.0,15.0,10.0,7.0,2.0
4,2026-03-05,6.0,14.0,10.0,7.0,2.0
5,2026-03-06,5.0,12.0,8.0,6.0,1.0
6,2026-03-07,3.0,6.0,4.0,3.0,1.0
7,2026-03-08,3.0,7.0,5.0,3.0,1.0
8,2026-03-09,7.0,16.0,11.0,7.0,2.0
9,2026-03-10,7.0,16.0,11.0,7.0,2.0
